# Multitask fine-tuning: MedText, PubMedQA and MedMCQA

Second stage of the pipeline. One model is fine-tuned on three medical tasks at once, starting either from the original GPT-2 or from the PubMed-pretrained GPT-2 (`pretrain_pubmed.ipynb`). Comparing the two answers the main question of the project: **does domain pretraining on PubMed help the downstream tasks?**

| Task | What the model learns | Test set | Metric |
|---|---|---|---|
| MedText | write the reference answer to a clinical question | MedText test (142) | answer-only loss |
| PubMedQA | answer yes / no / maybe from an abstract, then explain | official test set (500) | accuracy, macro F1 |
| MedMCQA | pick A-D for an exam question, then explain | official validation set (4,181) | accuracy, macro F1 |

Training and evaluation are tracked in MLflow (experiment `gpt2-med-multitask`).

In [ ]:
import os
from pathlib import Path

# The notebooks live in notebooks/; run from the repository root so `src` imports
# and relative paths (configs/, data/, checkpoints/, mlflow.db) resolve as in the CLIs
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

In [2]:
import json
from pathlib import Path

import pandas as pd

from src.config import load_multitask_config
from src.tracking import configure_tracking

MODEL = "gpt2"  # a file in configs/model/
config = load_multitask_config([f"model={MODEL}"])
configure_tracking(config)

## 1. Data

`prepare_multitask()` (`src/multitask.py`) turns every dataset into prompt-completion pairs with a shared format, and builds the splits:

- **PubMedQA**: the official 500 test questions; the other 500 labelled questions are split into 450 train and 50 validation, stratified by decision. The artificial set (PQA-A, 211k questions, 93% "yes") is used for training only, balanced by decision.
- **MedMCQA**: 5% of the train split is held out for model selection; the official validation set is the test set (the official test set has no answers). About half of the questions have an explanation, which is added after the answer letter.
- **Mixture**: per epoch, MedText and PubMedQA-labelled repeated 5 times, 15,000 "yes" + 15,000 "no" PQA-A questions, and 60,000 new MedMCQA questions (about 98,000 examples). The training set holds 5 such epochs.
- Examples longer than the 1,024-token context are dropped (a few long MedMCQA and PQA-A questions, and 2 MedMCQA test questions).

In [5]:
from src.multitask import load_multitask, prepare_multitask

prepare_multitask(config)
stats = json.loads((Path(config.multitask.processed_dir) / "stats.json").read_text())
splits = load_multitask(config.multitask.processed_dir)

pd.Series(stats["rows"], name="examples").to_frame()

data/processed/multitask_v1 already exists


,examples
train,489475
dev_combined,341
dev_medtext,141
dev_pubmedqa,50
dev_medmcqa,150
test_medtext,142
test_pubmedqa,500
test_medmcqa,4181


In [3]:
print(f"Examples per mixture epoch: {stats['examples_per_epoch']:,}")
print("Training examples by task (all epochs):", stats["train_tasks"])
for name, counts in stats["test_labels"].items():
    majority = max(counts.values()) / sum(counts.values())
    print(f"{name}: {counts} -> always answering the most frequent label scores {majority:.1%}")

Examples per mixture epoch: 97,895
Training examples by task (all epochs): {'medtext': 28225, 'pubmedqa': 161250, 'medmcqa': 300000}
test_pubmedqa: {'maybe': 55, 'no': 169, 'yes': 276} -> always answering the most frequent label scores 55.2%
test_medmcqa: {'A': 1348, 'B': 1084, 'C': 924, 'D': 825} -> always answering the most frequent label scores 32.2%


### One example per task

In [4]:
for task in ("medtext", "pubmedqa", "medmcqa"):
    example = splits[f"dev_{task}"][0]
    print(f"===== {task} =====")
    print(example["prompt"][-600:] + example["completion"][:300])
    print()

===== medtext =====
Question:
A knee X-ray of a 60-year-old female patient with chronic knee pain shows joint space narrowing and bony spurs around the joint. What is the most likely diagnosis?

Answer:
These findings are characteristic of osteoarthritis, a degenerative joint disease that frequently affects weight-bearing joints like the knees. It is characterized by the breakdown of cartilage, leading to bone remodeling, joint space narrowing, and osteophyte (bony spur) formation. Management is us

===== pubmedqa =====
, years' effects and regions' effects represented control variables.
RESULTS: Empirical estimates from instrumental variables suggest that the enactment of the traffic law reform in interaction with number of traffic tickets per police officer is significantly associated with a decrease of 8% in traffic fatalities and 7% in severe injuries. Piecewise regression model results for the 2007-2012 period suggest that police traffic enforcement reduced traffic fatalities by 5

## 2. Zero-shot baselines

Before any fine-tuning: the original GPT-2 and the PubMed-pretrained GPT-2 on the three test sets. For PubMedQA and MedMCQA the model scores each possible answer (the log-probability of `yes`/`no`/`maybe` or `A`-`D` right after the prompt) and the most likely one is its prediction; this works even for models that cannot follow the answer format yet.

In [5]:
from src.multitask_train import evaluate_model
from src.tracking import log_evaluation

pretrained_uri = config.finetune.dapt_model_id
baselines = {}
for name, model_id in [("GPT-2", config.model.model_id), ("GPT-2 + PubMed", pretrained_uri)]:
    baselines[name] = evaluate_model(model_id, config, prefix="test").metrics
    log_evaluation(model_id, "zero_shot", baselines[name], config)

pd.DataFrame(baselines).T.round(4)

Truncating train dataset: 100%|██████████| 142/142 [00:00<00:00, 31230.20 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 142/142 [00:00<00:00, 84842.05 examples/s]
/home/hpc3/Documents/Projects/medllm-finetuning/.venv/lib/python3.11/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)


Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
No log,2.904179,0,3.077652,0.429037,0.000000


2026/11/02 19:10:11 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/11/02 19:10:11 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/11/02 19:10:11 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
Truncating train dataset: 100%|██████████| 142/142 [00:00<00:00, 4204.46 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 142/142 [00:00<00:00, 87330.08 examples/s]


Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
No log,2.737820,0,2.710360,0.442050,0.000000


2026/11/02 19:10:27 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/11/02 19:10:27 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/11/02 19:10:27 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


,test_medtext_loss,test_pubmedqa_accuracy,test_pubmedqa_macro_f1,test_medmcqa_accuracy,test_medmcqa_macro_f1
GPT-2,2.9042,0.346,0.1789,0.3224,0.1244
GPT-2 + PubMed,2.7378,0.348,0.1870,0.3227,0.1296


## 3. Hyperparameter search (Optuna, run in a terminal)

The learning rate and batch size selected for MedText alone (1,129 examples) are not necessarily good for a mixture of ~98,000 examples per epoch, so they are searched again with Optuna before the full runs:

- **Search space** (`sweep` in the YAML): learning rate `1e-5`-`3e-4` (log scale), batch size 8/16/32, constant or cosine schedule, 0% or 5% warmup; 12 trials with the TPE sampler.
- **Budget**: each trial trains for one mixture epoch and is evaluated 4 times on the combined dev set; the median pruner stops trials that are worse than earlier ones at the same point. The learning-rate schedule has the length of a full run, so a trial's epoch matches the first epoch of the final run.
- **Fairness**: the search always uses the original GPT-2, and both base models are then fine-tuned with the same values.
- **No test data**: trials only see the dev sets.

```bash
nohup uv run python -m src.multitask_train --sweep model=gpt2 > logs/multitask_sweep.log 2>&1 &
```

Then copy the best values into the YAML: `learning_rate` and `batch_size` under `finetune`, `lr_scheduler_type` and `warmup_ratio` under `training`. Keeping them in the config (and in git) makes the final runs reproducible.

In [ ]:
sweep_rows = json.loads(Path(config.sweep.results_path).read_text())
pd.DataFrame(sweep_rows).sort_values("best_dev_combined_loss").round(6)

## 4. Fine-tuning (run in a terminal)

Each command fine-tunes the three seeds in `finetune.seeds` with the hyperparameters chosen in section 3 and logs one `multitask-<base>` MLflow run with a nested run per seed. Training stops when the combined dev loss has not improved for 2 mixture epochs; the best epoch is scored on the test sets. The best seed is registered as `models:/gpt2-med-multitask@<base>-best`.

From the original GPT-2:

```bash
nohup uv run python -m src.multitask_train --base original model=gpt2 > logs/multitask_gpt2.log 2>&1 &
```

From the PubMed-pretrained GPT-2 (`finetune.dapt_model_id`, the `candidate` alias):

```bash
nohup uv run python -m src.multitask_train --base dapt model=gpt2 > logs/multitask_dapt.log 2>&1 &
```

Run them one after the other: both use the GPU. In the DVC pipeline they are the `finetune_original@gpt2` and `finetune_dapt@gpt2` stages, which also write `results/gpt2/finetune_<base>.json`.

## 5. Results

Mean and standard deviation over the three seeds, read from the `multitask-<base>` runs in MLflow. A difference between the two base models matters only if it is clearly larger than the standard deviations.

In [3]:
import mlflow

experiment = mlflow.get_experiment_by_name(config.mlflow.experiment_name)
summary_runs = mlflow.search_runs(
    [experiment.experiment_id],
    filter_string="tags.task = 'multitask' AND tags.stage = 'finetune'",
    order_by=["start_time DESC"],
    output_format="list",
)
# The summary runs are the parents: they have no seed tag and hold the *_mean metrics
summary_runs = [run for run in summary_runs if "finetune_seed" not in run.data.tags]

rows = {}
for run in summary_runs:
    base = run.data.tags["base_model"]
    if base in rows:
        continue  # keep the latest run per base model
    metrics = run.data.metrics
    rows[base] = {
        name: f"{metrics[f'{name}_mean']:.4f} ± {metrics[f'{name}_std']:.4f}"
        for name in (
            "test_medtext_loss",
            "test_pubmedqa_accuracy",
            "test_pubmedqa_macro_f1",
            "test_medmcqa_accuracy",
            "test_medmcqa_macro_f1",
        )
        if f"{name}_mean" in metrics
    }

pd.DataFrame(rows).T

,test_medtext_loss,test_pubmedqa_accuracy,test_pubmedqa_macro_f1,test_medmcqa_accuracy,test_medmcqa_macro_f1
models:/gpt2-pubmed-dapt@candidate,1.9761 ± 0.0115,0.6513 ± 0.0234,0.4552 ± 0.0123,0.2983 ± 0.0316,0.2097 ± 0.0098
openai-community/gpt2,2.0341 ± 0.0091,0.6380 ± 0.0171,0.4470 ± 0.0089,0.3035 ± 0.0284,0.2213 ± 0.0208


## 6. Inference: what do the fine-tuned models answer?

The best seed of each run, loaded from the Model Registry. Choose a test question (`task` and `index`) or write your own `question` (it gets the MedText `Question:/Answer:` template). Answers use greedy decoding.

For PubMedQA and MedMCQA the line `scored choice` is the answer used in the metrics: the most likely of `yes`/`no`/`maybe` or `A`-`D` right after the prompt. It can differ from the first word of the generated text, which is only the single most likely continuation.

In [6]:
from transformers import AutoTokenizer

from src.choice_eval import choice_scores
from src.inference import generate_completion
from src.formatting import format_prompt
from src.modeling import load_model

finetuned_uris = {
    "GPT-2": f"models:/{config.mlflow.registered_model_name}@original-best",
    "GPT-2 + PubMed": f"models:/{config.mlflow.registered_model_name}@dapt-best",
}
tokenizer = AutoTokenizer.from_pretrained(config.model.model_id)
finetuned_models = {}
for name, uri in finetuned_uris.items():
    model, device = load_model(uri)
    model.config.use_cache = True  # load_model turns the cache off for training; generation is faster with it
    finetuned_models[name] = model

test_splits = load_multitask(config.multitask.processed_dir)
choices_by_task = {"pubmedqa": ["yes", "no", "maybe"], "medmcqa": ["A", "B", "C", "D"]}

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 27606.38it/s]


Device: cuda
Parameters: 124,439,808


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 24115.50it/s]

Device: cuda
Parameters: 124,439,808


In [15]:
task = "medtext"  # "medtext", "pubmedqa" or "medmcqa"
index = 0
question = None  # or your own question, e.g. "What are the first-line treatments for hypertension?"
max_new_tokens = 200

if question is None:
    example = test_splits[f"test_{task}"][index]
    prompt, reference = example["prompt"], example["completion"]
else:
    task, prompt, reference = "medtext", format_prompt(question), None

print(f"PROMPT:\n{prompt}")
if reference is not None:
    print(f"REFERENCE:\n{reference}\n")

for name, model in finetuned_models.items():
    print("=" * 80)
    print(f"{name}:")
    if task in choices_by_task:
        choices = choices_by_task[task]
        scores = choice_scores(model, tokenizer, [prompt], choices, device, batch_size=len(choices))[0]
        print(f"scored choice: {choices[scores.index(max(scores))]}")
    print(generate_completion(model, tokenizer, prompt, device, max_new_tokens))

PROMPT:
Question:
A 60-year-old man with a history of angina is now experiencing more frequent and severe angina attacks that are triggered by less exertion than before. What could this signify?

Answer:

REFERENCE:
This could indicate that the patient's stable angina is progressing towards unstable angina. The increased frequency, severity, and occurrence of angina at lower levels of exertion suggest worsening coronary artery disease. The patient should be urgently evaluated, which might include an ECG, cardiac biomarkers, and possibly coronary angiography, to assess the severity of the disease and to guide treatment.

GPT-2:
This patient's symptoms suggest a possible diagnosis of angina, a complication of chronic stress. This condition is characterized by increased blood pressure, increased blood flow, and increased blood pressure in the circulation. This is known as angina syndrome. The patient's symptoms could suggest a complication of chronic stress, such as coronary artery diseas

## 7. LLM-as-judge (G-Eval) on the MedText answers

The MedText loss says how likely the reference answer is under the model, not whether the generated answer is clinically right. Here every seed's best model answers the 142 MedText test questions, and a local judge model (Qwen3.8-27B, served by vLLM) scores each answer against the reference from 0 to 1 (`src/geval.py`):

| Criterion | Question for the judge |
|---|---|
| correctness | same main conclusion (diagnosis, explanation, action) as the reference, with no contradictions or invented facts? |
| completeness | which key clinical points of the reference (cause, tests, treatment, urgency) does it cover? |
| coherence | does it answer the question, without contradicting or repeating itself? |

The stage first generates every answer, then starts the judge server with vLLM (installed apart from the project with `uv tool install vllm`; settings in the `geval` section of `configs/config.yaml`, log in `logs/vllm.log`), scores the answers and stops the server. A server already running at `geval.base_url` is used instead. The scores go to the seed runs (with a table of every answer and the judge's reasons) and their mean and std to the `multitask-<base>` runs.

```bash
nohup uv run python -m src.geval model=gpt2 > logs/geval.log 2>&1 &
```

In the DVC pipeline this is the `geval@gpt2` stage, which also writes `results/gpt2/geval.json`.

Before trusting the scores, read some answers below with the judge's reasons and check you agree with them.

In [ ]:
import mlflow

from src.geval import GEVAL_METRICS, METRIC_PREFIX
from src.report import BASES, load_stage_result

geval = load_stage_result(config.results_dir, "geval")
geval_rows = {}
answer_tables = {}
for base in BASES:
    summary = geval["bases"][base]
    geval_rows[base] = {}
    for name in GEVAL_METRICS:
        score = summary[f"{METRIC_PREFIX}_{name}"]
        geval_rows[base][name] = f"{score['mean']:.3f} ± {score['std']:.3f}"

    first_seed = load_stage_result(config.results_dir, f"finetune_{base}")["seeds"][0]
    answer_tables[f"{base} (seed {first_seed['seed']})"] = mlflow.load_table(
        "geval/test_medtext.json", run_ids=[first_seed["run_id"]]
    )

pd.DataFrame(geval_rows).T

In [ ]:
index = 0  # MedText test question

first_table = next(iter(answer_tables.values()))
print(f"QUESTION:\n{first_table.loc[index, 'prompt']}")
print(f"REFERENCE:\n{first_table.loc[index, 'reference']}\n")
for name, table in answer_tables.items():
    row = table.loc[index]
    print("=" * 80)
    print(f"{name}:\n{row['answer']}\n")
    for criterion in GEVAL_METRICS:
        print(f"- {criterion} {row[criterion]:.2f}: {row[f'{criterion}_reason']}")